In [ ]:
from __future__ import annotations

import os
import csv
import pickle
from datetime import datetime
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd

In [ ]:
# ============================================================
# User settings
# ============================================================

def resolve_repository_root() -> Path:
    root = Path(
        os.environ.get(
            "PHYLOGENY_REPOSITORY_ROOT",
            Path.cwd(),
        )
    ).expanduser().resolve()

    if root.name in {"AA", "NT", "common"}:
        root = root.parent

    return root


REPOSITORY_ROOT = resolve_repository_root()

SEQUENCE_TYPE = "AA"

PROJECT_ROOT = REPOSITORY_ROOT / "AA"

DATA_ROOT = PROJECT_ROOT / "data"

MATRIX_DIR = DATA_ROOT / "matrices"
TREE_DIR = DATA_ROOT / "trees"

SIMULATION_MANIFEST = (
    DATA_ROOT
    / "manifests"
    / "simulation_manifest.csv"
)

RECONSTRUCT_DIR = (
    PROJECT_ROOT
    / "results_reconstruct"
)

EVALUATION_DIR = (
    PROJECT_ROOT
    / "results_evaluation"
)


INCLUDE_GENERATORS = {
    "rtree",
    "yule",
}

INCLUDE_BLS = {
    "0.125",
    "0.250",
    "0.500",
    "0.625",
    "0.750",
}

INCLUDE_REPS = set(range(1, 101))


VARIANTS = (
    "baseline_nmcut",
    "logkernel_nmcut",
    "poisson20_selftune_nmcut",
    "wag_selftune_nmcut",
    "logglobal_nmcut",
)

POSTSWAP_MODES = (
    "none",
    "ncut",
)

# Neighbor-joining trees produced by ape::nj()
NJ_DIR = (
    PROJECT_ROOT
    / "results_nj"
)

NJ_VARIANTS = (
    "log_nj",
    "er20_nj",
    "wag_nj",
)


In [ ]:
def load_pickle(path: Path) -> Any:
    with path.open("rb") as file:
        return pickle.load(file)


def read_manifest(
    path: Path,
) -> list[dict[str, str]]:
    rows = []

    with path.open(newline="") as file:
        reader = csv.DictReader(file)

        for row in reader:
            if row.get("status") == "ok":
                rows.append(row)

    return rows


def parse_tag(
    tag: str,
) -> dict[str, Any]:
    # Example:
    # rtree_n30_bl0.125_rep001

    parts = tag.split("_")

    if len(parts) != 4:
        raise ValueError(
            f"unexpected tag format: {tag}"
        )

    return {
        "generator": parts[0],
        "n_taxa": int(parts[1][1:]),
        "bl": parts[2][2:],
        "rep": int(parts[3][3:]),
    }


def should_select_tag(
    tag: str,
) -> bool:
    info = parse_tag(tag)

    return (
        info["generator"] in INCLUDE_GENERATORS
        and info["bl"] in INCLUDE_BLS
        and info["rep"] in INCLUDE_REPS
    )


# ============================================================
# Minimal Newick parser
# ============================================================

class NwkNode:
    def __init__(
        self,
        name: Optional[str] = None,
        children: Optional[list["NwkNode"]] = None,
    ):
        self.name = name
        self.children = children or []

    @property
    def is_leaf(self) -> bool:
        return len(self.children) == 0


def skip_spaces(
    text: str,
    index: int,
) -> int:
    while (
        index < len(text)
        and text[index].isspace()
    ):
        index += 1

    return index


def read_label(
    text: str,
    index: int,
) -> tuple[str, int]:
    index = skip_spaces(
        text,
        index,
    )

    start = index

    while (
        index < len(text)
        and text[index] not in "(),:;"
    ):
        index += 1

    return (
        text[start:index].strip(),
        index,
    )


def skip_branch_length(
    text: str,
    index: int,
) -> int:
    index = skip_spaces(
        text,
        index,
    )

    if (
        index < len(text)
        and text[index] == ":"
    ):
        index += 1

        while (
            index < len(text)
            and text[index] not in ",);"
        ):
            index += 1

    return index


def parse_subtree(
    text: str,
    index: int,
) -> tuple[NwkNode, int]:
    index = skip_spaces(
        text,
        index,
    )

    if text[index] == "(":
        index += 1

        children = []

        while True:
            child, index = parse_subtree(
                text,
                index,
            )

            children.append(child)

            index = skip_spaces(
                text,
                index,
            )

            if text[index] == ",":
                index += 1
                continue

            if text[index] == ")":
                index += 1
                break

            raise ValueError(
                "unexpected Newick character: "
                f"{text[index]}"
            )

        label, index = read_label(
            text,
            index,
        )

        index = skip_branch_length(
            text,
            index,
        )

        return (
            NwkNode(
                name=label or None,
                children=children,
            ),
            index,
        )

    label, index = read_label(
        text,
        index,
    )

    if not label:
        raise ValueError(
            "empty leaf label"
        )

    index = skip_branch_length(
        text,
        index,
    )

    return (
        NwkNode(
            name=label,
        ),
        index,
    )


def parse_newick(
    text: str,
) -> NwkNode:
    text = text.strip()

    if not text.endswith(";"):
        text += ";"

    root, index = parse_subtree(
        text,
        0,
    )

    index = skip_spaces(
        text,
        index,
    )

    if (
        index >= len(text)
        or text[index] != ";"
    ):
        raise ValueError(
            "unexpected trailing Newick content"
        )

    return root


# ============================================================
# Split extraction
# ============================================================

def canonical_split(
    split: set[int] | frozenset[int],
    n: int,
) -> frozenset[int]:
    side = frozenset(
        int(x)
        for x in split
    )

    complement = frozenset(
        set(range(n)) - set(side)
    )

    if len(side) < len(complement):
        return side

    if len(complement) < len(side):
        return complement

    side_key = tuple(
        sorted(side)
    )

    complement_key = tuple(
        sorted(complement)
    )

    return (
        side
        if side_key <= complement_key
        else complement
    )


def collect_leaf_names(
    node: NwkNode,
) -> set[str]:
    if node.is_leaf:
        if node.name is None:
            raise ValueError(
                "leaf without name"
            )

        return {node.name}

    leaves = set()

    for child in node.children:
        leaves |= collect_leaf_names(
            child
        )

    return leaves


def collect_name_splits(
    node: NwkNode,
    all_leaves: set[str],
    splits: set[frozenset[str]],
) -> set[str]:
    if node.is_leaf:
        if node.name is None:
            raise ValueError(
                "leaf without name"
            )

        return {node.name}

    current = set()

    for child in node.children:
        current |= collect_name_splits(
            child,
            all_leaves,
            splits,
        )

    complement = all_leaves - current

    if (
        len(current) >= 2
        and len(complement) >= 2
    ):
        if len(current) < len(complement):
            canonical = frozenset(
                current
            )

        elif len(complement) < len(current):
            canonical = frozenset(
                complement
            )

        else:
            current_key = tuple(
                sorted(current)
            )

            complement_key = tuple(
                sorted(complement)
            )

            canonical = (
                frozenset(current)
                if current_key <= complement_key
                else frozenset(complement)
            )

        splits.add(canonical)

    return current


def reference_splits(
    tree_path: Path,
    ids: list[str],
) -> set[frozenset[int]]:
    root = parse_newick(
        tree_path.read_text()
    )

    all_leaves = collect_leaf_names(
        root
    )

    if all_leaves != set(ids):
        raise ValueError(
            f"leaf mismatch: {tree_path}"
        )

    name_splits: set[
        frozenset[str]
    ] = set()

    collect_name_splits(
        root,
        all_leaves,
        name_splits,
    )

    name_to_index = {
        name: index
        for index, name in enumerate(ids)
    }

    output = set()

    for split in name_splits:
        index_split = {
            name_to_index[name]
            for name in split
        }

        output.add(
            canonical_split(
                index_split,
                len(ids),
            )
        )

    return output


def reconstruction_splits(
    payload: dict[str, Any],
    n: int,
) -> set[frozenset[int]]:
    output = set()

    for split in payload.get(
        "internal_splits",
        [],
    ):
        canonical = canonical_split(
            set(int(x) for x in split),
            n,
        )

        if (
            2
            <= len(canonical)
            <= n - 2
        ):
            output.add(canonical)

    return output


# ============================================================
# Metrics
# ============================================================

def compute_rf_metrics(
    ref_splits: set[frozenset[int]],
    pred_splits: set[frozenset[int]],
    *,
    n_leaves: int,
) -> dict[str, float]:
    tp = len(
        ref_splits & pred_splits
    )

    n_ref = len(
        ref_splits
    )

    n_pred = len(
        pred_splits
    )

    fp = n_pred - tp
    fn = n_ref - tp

    rf_distance = fp + fn

    max_rf = max(
        2 * n_leaves - 6,
        1,
    )

    recovered_percent = (
        1.0
        - rf_distance / max_rf
    ) * 100.0

    recovered_branch_percent = (
        tp / n_ref * 100.0
        if n_ref > 0
        else np.nan
    )

    precision = (
        tp / n_pred
        if n_pred > 0
        else np.nan
    )

    recall = (
        tp / n_ref
        if n_ref > 0
        else np.nan
    )

    if (
        precision == 0
        or recall == 0
        or np.isnan(precision)
        or np.isnan(recall)
    ):
        f1 = 0.0

    else:
        f1 = (
            2.0
            * precision
            * recall
            / (precision + recall)
        )

    return {
        "rf_distance": float(
            rf_distance
        ),
        "max_rf": float(
            max_rf
        ),
        "recovered_percent": float(
            recovered_percent
        ),
        "recovered_branch_percent": float(
            recovered_branch_percent
        ),
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
        "n_ref_splits": float(
            n_ref
        ),
        "n_pred_splits": float(
            n_pred
        ),
        "n_true_positive_splits": float(
            tp
        ),
    }


# ============================================================
# Evaluation
# ============================================================

def evaluate_qubo_all() -> pd.DataFrame:
    manifest_rows = read_manifest(
        SIMULATION_MANIFEST
    )

    tags = sorted(
        row["tag"]
        for row in manifest_rows
        if should_select_tag(
            row["tag"]
        )
    )

    print(
        f"[info] selected tags: {len(tags)}"
    )

    rows = []

    for tag in tags:
        info = parse_tag(tag)

        matrix_path = (
            MATRIX_DIR
            / f"{tag}_matrices.pkl"
        )

        tree_path = (
            TREE_DIR
            / f"{tag}.nwk"
        )

        for variant in VARIANTS:
            for postswap_mode in (
                POSTSWAP_MODES
            ):
                base_row = {
                    "sequence_type": (
                        SEQUENCE_TYPE
                    ),
                    "method": "SQBM",
                    "tag": tag,
                    "generator": (
                        info["generator"]
                    ),
                    "bl": float(
                        info["bl"]
                    ),
                    "rep": info["rep"],
                    "n_taxa": info["n_taxa"],
                    "variant": variant,
                    "postswap_mode": (
                        postswap_mode
                    ),
                }

                if not matrix_path.exists():
                    rows.append(
                        {
                            **base_row,
                            "status": (
                                "missing_matrix"
                            ),
                        }
                    )

                    continue

                if not tree_path.exists():
                    rows.append(
                        {
                            **base_row,
                            "status": (
                                "missing_true_tree"
                            ),
                        }
                    )

                    continue

                reconstruction_path = (
                    RECONSTRUCT_DIR
                    / variant
                    / postswap_mode
                    / (
                        f"reconstruction_"
                        f"{tag}.pkl"
                    )
                )

                if (
                    not reconstruction_path.exists()
                ):
                    rows.append(
                        {
                            **base_row,
                            "status": (
                                "missing_reconstruction"
                            ),
                        }
                    )

                    continue

                try:
                    matrix_payload = load_pickle(
                        matrix_path
                    )

                    ids = list(
                        matrix_payload["ids"]
                    )

                    ref_splits = reference_splits(
                        tree_path,
                        ids,
                    )

                    reconstruction = load_pickle(
                        reconstruction_path
                    )

                    pred_splits = (
                        reconstruction_splits(
                            reconstruction,
                            len(ids),
                        )
                    )

                    metrics = compute_rf_metrics(
                        ref_splits,
                        pred_splits,
                        n_leaves=len(ids),
                    )

                    rows.append(
                        {
                            **base_row,
                            "status": "ok",
                            **metrics,
                            "total_ncut": (
                                reconstruction.get(
                                    "total_ncut",
                                    np.nan,
                                )
                            ),
                            "fallback_count": (
                                reconstruction.get(
                                    "fallback_count",
                                    np.nan,
                                )
                            ),
                            "total_postswaps": (
                                reconstruction.get(
                                    "total_postswaps",
                                    np.nan,
                                )
                            ),
                        }
                    )

                except Exception as error:
                    rows.append(
                        {
                            **base_row,
                            "status": "error",
                            "message": (
                                f"{type(error).__name__}: "
                                f"{error}"
                            ),
                        }
                    )

    return pd.DataFrame(rows)



def evaluate_nj_all() -> pd.DataFrame:
    """Evaluate NJ Newick trees using the same split code as SQBM trees."""
    manifest_rows = read_manifest(
        SIMULATION_MANIFEST
    )

    tags = sorted(
        row["tag"]
        for row in manifest_rows
        if should_select_tag(
            row["tag"]
        )
    )

    print(
        f"[info] selected NJ tags: {len(tags)}"
    )

    rows = []

    for tag in tags:
        info = parse_tag(tag)

        matrix_path = (
            MATRIX_DIR
            / f"{tag}_matrices.pkl"
        )

        true_tree_path = (
            TREE_DIR
            / f"{tag}.nwk"
        )

        for variant in NJ_VARIANTS:
            base_row = {
                "sequence_type": SEQUENCE_TYPE,
                "method": "NJ",
                "tag": tag,
                "generator": info["generator"],
                "bl": float(info["bl"]),
                "rep": info["rep"],
                "n_taxa": info["n_taxa"],
                "variant": variant,
                "postswap_mode": "not_applicable",
            }

            if not matrix_path.exists():
                rows.append(
                    {
                        **base_row,
                        "status": "missing_matrix",
                    }
                )
                continue

            if not true_tree_path.exists():
                rows.append(
                    {
                        **base_row,
                        "status": "missing_true_tree",
                    }
                )
                continue

            nj_tree_path = (
                NJ_DIR
                / variant
                / f"{tag}.nwk"
            )

            if not nj_tree_path.exists():
                rows.append(
                    {
                        **base_row,
                        "status": "missing_nj_tree",
                    }
                )
                continue

            try:
                matrix_payload = load_pickle(
                    matrix_path
                )

                ids = [
                    str(x)
                    for x in matrix_payload["ids"]
                ]

                ref_splits = reference_splits(
                    true_tree_path,
                    ids,
                )

                # The same Newick-to-split function is used for the
                # predicted NJ tree, ensuring identical evaluation.
                pred_splits = reference_splits(
                    nj_tree_path,
                    ids,
                )

                metrics = compute_rf_metrics(
                    ref_splits,
                    pred_splits,
                    n_leaves=len(ids),
                )

                rows.append(
                    {
                        **base_row,
                        "status": "ok",
                        **metrics,
                        "total_ncut": np.nan,
                        "fallback_count": np.nan,
                        "total_postswaps": np.nan,
                    }
                )

            except Exception as error:
                rows.append(
                    {
                        **base_row,
                        "status": "error",
                        "message": (
                            f"{type(error).__name__}: "
                            f"{error}"
                        ),
                    }
                )

    return pd.DataFrame(rows)


def evaluate_all() -> pd.DataFrame:
    qubo_df = evaluate_qubo_all()
    nj_df = evaluate_nj_all()

    output = pd.concat(
        [qubo_df, nj_df],
        ignore_index=True,
        sort=False,
    )

    return (
        output
        .sort_values(
            [
                "method",
                "generator",
                "bl",
                "rep",
                "variant",
                "postswap_mode",
            ]
        )
        .reset_index(drop=True)
    )

def make_summary(
    evaluation_df: pd.DataFrame,
) -> pd.DataFrame:
    ok_df = (
        evaluation_df[
            evaluation_df["status"] == "ok"
        ]
        .copy()
    )

    if ok_df.empty:
        return pd.DataFrame()

    return (
        ok_df
        .groupby(
            [
                "method",
                "generator",
                "bl",
                "variant",
                "postswap_mode",
            ],
            as_index=False,
        )
        .agg(
            n=("tag", "count"),
            rf_mean=(
                "rf_distance",
                "mean",
            ),
            rf_sd=(
                "rf_distance",
                "std",
            ),
            recovered_mean=(
                "recovered_percent",
                "mean",
            ),
            recovered_sd=(
                "recovered_percent",
                "std",
            ),
            recovered_branch_mean=(
                "recovered_branch_percent",
                "mean",
            ),
            recovered_branch_sd=(
                "recovered_branch_percent",
                "std",
            ),
            precision_mean=(
                "precision",
                "mean",
            ),
            recall_mean=(
                "recall",
                "mean",
            ),
            f1_mean=(
                "f1",
                "mean",
            ),
        )
        .sort_values(
            [
                "method",
                "generator",
                "bl",
                "variant",
                "postswap_mode",
            ]
        )
        .reset_index(drop=True)
    )


In [ ]:
EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

run_timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

evaluation_df = evaluate_all()

output_path = (
    EVALUATION_DIR
    / (
        f"evaluation_"
        f"{SEQUENCE_TYPE.lower()}_"
        f"{run_timestamp}.csv"
    )
)

evaluation_df.to_csv(
    output_path,
    index=False,
)

summary_df = make_summary(
    evaluation_df
)

n_ok = int(
    (evaluation_df["status"] == "ok").sum()
)

n_non_ok = len(evaluation_df) - n_ok

print(
    f"[info] total rows: {len(evaluation_df)}"
)

print(
    f"[info] OK rows: {n_ok}"
)

print(
    f"[info] non-OK rows: {n_non_ok}"
)

print(
    f"[saved] {output_path}"
)

print(
    "\n== Summary by condition =="
)

display(summary_df)

print(
    "\n== Status by method =="
)

display(
    evaluation_df
    .groupby(
        ["method", "status"],
        as_index=False,
    )
    .size()
)
